# Appendix — Microsoft Agent Framework: the same triage, gated

The [Microsoft Agent Framework](https://learn.microsoft.com/en-us/agent-framework/overview/) "combines AutoGen's simple agent abstractions with Semantic Kernel's enterprise features," plus graph-based workflows. This appendix runs the course's fixed ticket — **TKT-2205** — through it: same tools, same world data, same economics you have hit in every other appendix. The point is not a new agent idea. It is to line up its `Agent` (the framework's *ChatAgent*), its `@tool` functions, and its native function-approval gate, one for one, against the loop, registry, and controls you built by hand in Parts 1-3.

> **Before running this notebook:** `pip install -e ".[msaf]"` (once). That extra installs **agent-framework-core** into the main venv — no separate kernel. Do *not* reach for the full `agent-framework` metapackage: it pulls Azure / Foundry client extras this appendix never touches. Core ships the `Agent`, the tool and middleware machinery, and the `BaseChatClient` base class — but **no concrete model client** (the OpenAI one lives in a separate `agent-framework-openai` package), so we reach OpenRouter through a tiny `litellm`-backed client built below. Everything else stays the same.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

## The invariant: what TKT-2205 must decide

Every framework appendix ends at the same numbers so they are comparable. The gold is not hand-typed — it comes straight from `shoplab.rules.decide`, the deterministic cascade the whole world is graded against. For TKT-2205 (an *opened* boot returned by a *non-vip member*, 18 days out, inside the 30-day window) branch 9 fires: item value minus a 10% restocking fee, `$189.99 x 0.90`.

In [ ]:
# The one fixed decision this appendix must land on (authoritative).
from shoplab import world
from shoplab.rules import decide

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}
ticket = next(t for t in world.load_tickets()["train"] if t["ticket_id"] == "TKT-2205")
order, customer = orders[ticket["order_id"]], customers[ticket["customer_id"]]

GOLD = decide(ticket, order, customer)
print("TKT-2205 gold:", GOLD)          # partial_refund / pol-restocking / 170.99
print("arithmetic:", 189.99, "x 0.90 =", round(189.99 * 0.90, 2))

## The model boundary is the one piece you build

Agent Framework's `Agent` is its *ChatAgent*: give it a chat client, instructions, and tools, and `run` drives the model-call / tool-call loop for you — the job `run_agent` did in ch02. But core ships no concrete model client, so the model boundary is the one thing we hand-build: a `BaseChatClient` subclass whose single required method, `_inner_get_response`, translates the framework's messages and tools to the OpenAI wire format, calls `litellm.completion`, and translates the reply back. Routing through `litellm` keeps the same `openrouter/` model string and the same disk cache from the config cell — reruns stay ~free. Mixing in `FunctionInvocationLayer` is what makes the client execute the tool calls and loop; without it the `Agent` would take only one turn.

In [ ]:
import json
from agent_framework import (BaseChatClient, Message, FunctionInvocationLayer,
                             Content, ChatResponse)

def _to_wire(messages, options):                       # framework messages -> OpenAI wire
    wire = ([{"role": "system", "content": options["instructions"]}]
            if options.get("instructions") else [])
    for m in messages:
        if str(m.role) == "tool":                      # tool results -> role:"tool" messages
            wire += [{"role": "tool", "tool_call_id": c.call_id, "content": str(c.result)}
                     for c in m.contents if c.type == "function_result"]
            continue
        text = "".join(c.text for c in m.contents if c.type == "text")
        calls = [{"id": c.call_id, "type": "function",
                  "function": {"name": c.name, "arguments": c.arguments
                               if isinstance(c.arguments, str) else json.dumps(c.arguments or {})}}
                 for c in m.contents if c.type == "function_call"]
        msg = {"role": str(m.role), "content": text or None}
        if calls:
            msg["tool_calls"] = calls
        wire.append(msg)
    return wire

class LiteLLMChatClient(FunctionInvocationLayer, BaseChatClient):
    """The ch01 model boundary as an Agent Framework client: litellm in, ChatResponse out."""
    def __init__(self, model, **kw):
        super().__init__(**kw)
        self._model = model

    async def _inner_get_response(self, *, messages, stream, options, **kwargs):
        tools = [t.to_json_schema_spec() for t in (options.get("tools") or [])]
        extra = {"tools": tools, "tool_choice": options.get("tool_choice", "auto")} if tools else {}
        m = litellm.completion(model=self._model, messages=_to_wire(messages, options),
                               temperature=TEMPERATURE, **extra).choices[0].message
        contents = [Content.from_text(m.content)] if m.content else []
        contents += [Content.from_function_call(tc.id, tc.function.name, arguments=tc.function.arguments)
                     for tc in (m.tool_calls or [])]
        return ChatResponse(messages=Message(role="assistant", contents=contents))

client = LiteLLMChatClient(MODEL)
print("client:", type(client).__name__, "| model:", MODEL)

## The four ops-desk tools, as `@tool` functions

`@tool` turns a plain annotated function into a tool — its signature becomes the JSON schema, its docstring the description — exactly what `to_openai_tools` did by hand over the ch02 `Tool` registry. The wrappers stay thin: each hands straight to `shoplab.tools.run_tool`, ch02's tolerant entry point (a bad call comes back as an `{"error": ...}` the model can read and recover from), so the world data and the risky-tool `Ledger` are the same objects the rest of the course uses. `issue_refund` carries `approval_mode="always_require"` — Agent Framework's built-in function-approval gate, the ch08 `require_approval` idea as one keyword.

In [ ]:
from agent_framework import tool
from shoplab.tools import standard_tools, Ledger, run_tool

ledger = Ledger()             # ch02's risky-tool audit log
_t = standard_tools(ledger)   # the 9 course tools, backed by the world data

@tool
def get_order(order_id: str) -> str:
    """Look up an order by id: items, totals, status, dates."""
    return run_tool(_t, "get_order", {"order_id": order_id})

@tool
def search_policy(query: str, k: int = 2) -> str:
    """Keyword-search the 12 store policy documents."""
    return run_tool(_t, "search_policy", {"query": query, "k": k})

@tool
def calc(expr: str) -> str:
    """Evaluate an arithmetic expression, e.g. '0.9 * 189.99'."""
    return run_tool(_t, "calc", {"expr": expr})

In [ ]:
@tool(approval_mode="always_require")   # the ch08 approval gate, declared on the tool
def issue_refund(order_id: str, amount_usd: float, reason: str) -> str:
    """Send money back to the customer. Irreversible."""
    return run_tool(_t, "issue_refund",
                    {"order_id": order_id, "amount_usd": amount_usd, "reason": reason})

TOOLS = [get_order, search_policy, calc, issue_refund]
agent = client.as_agent(name="ops-desk", tools=TOOLS, instructions=(
    "You are the Larkspur Outfitters ops-desk agent. Triage one return ticket end to end "
    "with your tools, in order: get_order for the unit price, search_policy for the governing "
    "policy, calc the refund (item value minus a 10% restocking fee, rounded to the nearest "
    "cent), then issue_refund for that exact amount. You MUST call issue_refund. Then state "
    "the decision (partial_refund), the policy id, and the dollar amount."))
print("agent:", agent.name, "| tools:", [t.name for t in TOOLS])

## Run the triage — the gate holds the refund

`agent.run` is the framework's loop: model call, tool calls, repeat. When the agent reaches `issue_refund`, `approval_mode="always_require"` stops the run and returns a **function-approval request** — a pending call awaiting a human — instead of moving money. We run it on a `session` so the paused state can be resumed. `run.user_input_requests` is that queue; the `Ledger` is still empty, because nothing has executed yet.

In [ ]:
TASK = (
    "Ticket TKT-2205: order ORD-7312, customer CUST-07 (member, non-vip), sku LK-1016 qty 1, "
    "condition opened, 18 days since delivery, requests a refund to the original payment method."
)
session = agent.create_session()
paused = await agent.run(TASK, session=session)

for req in paused.user_input_requests:
    print("APPROVAL NEEDED ->", req.function_call.name, req.function_call.arguments)
print("ledger (has the refund executed?):", ledger.entries)

> **What you should see:** one function-approval request for `issue_refund`, its arguments already carrying `amount_usd` = **170.99**, and the ledger **still empty**. The gate paused the loop the instant the agent reached for money — after it had read the order, found the policy, and computed the 10% fee, but before any of it moved.

## Approve, resume, and land

A human approves the pending call — `req.to_function_approval_response(True)` (passing `False` would block it and let the agent recover) — and we feed that back on the same `session`, so `agent.run` resumes from exactly where it stopped. Now `issue_refund` executes, the `Ledger` records it, and the agent finishes. We check the money it actually moved against the gold.

In [ ]:
approvals = [req.to_function_approval_response(True) for req in paused.user_input_requests]
result = await agent.run(Message(role="user", contents=approvals), session=session)

print("final answer:", (result.text or "").strip()[:220])
print("ledger:", ledger.entries)

moved = ledger.entries[-1]["amount_usd"]
print(f"agent moved ${moved}  |  gold ${GOLD['refund_usd']}  |  match: {moved == GOLD['refund_usd']}")

> **What you should see:** after approval the refund executes; the ledger holds a single `issue_refund` entry for **$170.99**, matching the gold exactly — the same **partial_refund / pol-restocking / 170.99** every appendix lands on (`$189.99 x 0.90`). The final prose names the decision; the auditable fact is the ledger entry. (No typed output schema is imposed here: with this model, forcing one tempts the agent to skip the gated tool and just assert the number, which would defeat the demo.)

## Machinery map: Agent Framework concept to the part you built

Line them up and the framework stops being magic — it is Parts 1-3, packaged with defaults and a runtime. Nothing in this appendix is a new idea about agents; it is the ops-desk loop you already wrote, wearing Agent Framework's names.

| Microsoft Agent Framework | Your hand-built equivalent | Built in |
|---|---|---|
| `Agent` (the ChatAgent) + `agent.run` | `run_agent`: model call -> execute tool calls -> repeat until done | ch02 |
| `@tool` (signature + docstring -> schema) | `to_openai_tools` over the `Tool` registry | ch02 |
| `FunctionInvocationLayer` driving the tool loop | the loop body that runs each tool call and re-prompts | ch02 |
| `BaseChatClient._inner_get_response` | `shoplab.llm.complete` over LiteLLM, same provider string | ch01 |
| `approval_mode="always_require"` + `to_function_approval_response` | `require_approval` wrapping a risky `Tool` in a gate | ch08 |
| `ChatMiddleware` / `FunctionMiddleware` (shipped, unused here) | `Budget` / `Checkpoint` hung off the loop's `on_step` hook | ch08 |
| `AgentSession` carrying the paused run | the message list you threaded through the loop by hand | ch02 |

## The honest read

Agent Framework gave us the loop (`Agent` + `FunctionInvocationLayer`), the schema plumbing (`@tool`), and a real approval gate (`approval_mode`) for free. What `agent-framework-core` did *not* give us is a model client — the OpenAI one is a separate package — so the model boundary is the single piece we still wrote by hand, and it turned out to be the ch01 `litellm` call in a new coat. What none of it changed is the hard part: the world, the tools, the rules, and the decision are still `shoplab`. An appendix, not a rewrite — the framework is a convenience over machinery you now understand well enough to have skipped it.